# 01 · Checking the simulated book

Run each cell with **Shift+Enter**. Cells under **Your turn** contain `...` placeholders; replace them, run, then run the check beneath. Checks print `PASS` or `NOT YET` and never stop the notebook. Worked answers are in the matching `_solutions` notebook.

The checks compare your results with the **truth in `python/simulate.py`**, so they keep working if you change the simulator's settings.

Before fitting any model, prove the simulator does what you designed. If these checks fail, every later result is suspect.

In [ ]:
def check(label, get, expected=True):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [ ]:
import sys
sys.path.insert(0, "../python")

import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import simulate as sim
importlib.reload(sim)

members = pd.read_parquet("../data/members.parquet")
print(f"{len(members):,} members, {members['lapsed'].mean():.1%} lapsed")

## 1. Headline numbers

### Your turn 1.1

Set `lapse_rate` (the share of members who lapsed) and `median_duration` (the median months observed).

In [ ]:
lapse_rate = ...
median_duration = ...

In [ ]:
check("1.1 lapse rate plausible", lambda: 0.2 < lapse_rate < 0.7)
check("1.1 everyone observed at least a month", lambda: members["duration"].min() >= 1)

## 2. The hazard by tenure

The empirical monthly hazard at tenure *t* is the share of members still active at *t* who lapse in that month: the mean of `lapse` across person-month rows at that tenure.

### Your turn 2.1

Build `pm` with `sim.person_months(members)`, then `hazard_by_tenure`: the mean of `lapse` grouped by `tenure`. Plot the first 60 months. You should see your design: higher early, a spike every 12 months, lower after 36.

In [ ]:
pm = ...
hazard_by_tenure = ...

In [ ]:
h = lambda t: hazard_by_tenure.loc[t]
check("2.1 anniversary spike at 12", lambda: h(12) > 1.8 * (h(11) + h(13)) / 2)
check("2.1 early months higher than months 7 to 11", lambda: hazard_by_tenure.loc[1:6].mean() > hazard_by_tenure.loc[7:11].mean())
check("2.1 long tenure lower", lambda: hazard_by_tenure.loc[38:46].mean() < hazard_by_tenure.loc[26:34].mean())

## 3. Actual vs expected: the actuarial sanity check

Your simulator knows each person-month's true hazard. Summing those gives the **expected** number of lapses; the observed count is the **actual**. If the simulator is right, A/E should be close to 1.

### Your turn 3.1

Add `true_hazard` to `pm` with `sim.hazard(sim.BASELINE + sim.member_eta(pm) + sim.tenure_eta(pm["tenure"]))`, then set `ae` to actual lapses divided by expected.

In [ ]:
# pm["true_hazard"] = ...
ae = ...

In [ ]:
check("3.1 A/E within 3% of 1", lambda: 0.97 < ae < 1.03)

## Wrap-up

If all checks pass, your simulator is trustworthy. Commit from VS Code: `Simulator checked: hazard shape and A/E`.